# ReLU and Leaky ReLU

Activation functions add non-linearity so the network can learn complex patterns.

```mermaid
flowchart LR
    I[Input] --> L1[Linear]
    L1 --> A[ReLU or Leaky ReLU]
    A --> L2[Linear]
    L2 --> O[Output]
```

Common choices:

- Hidden layers: `ReLU` or `LeakyReLU`
- Binary output: `Sigmoid`
- Multi-class probabilities: `Softmax`

## Why Not Sigmoid in Hidden Layers?

Sigmoid keeps outputs between `0` and `1`. At very negative or positive inputs, it becomes almost flat.

```text
flat function -> tiny gradient -> tiny parameter update
```

When many layers pass along tiny gradients, earlier layers may barely learn. This is the **vanishing-gradient problem**. Softmax can also saturate, so both are normally reserved for outputs rather than hidden layers.

> When using `CrossEntropyLoss`, pass raw logits. It handles the softmax calculation internally.

In [ ]:
import torch
from torch import nn

# requires_grad=True tells PyTorch to track calculations involving x
x_sigmoid = torch.tensor([-10.0, -2.0, 0.0, 2.0, 10.0], requires_grad=True)
sigmoid_output = torch.sigmoid(x_sigmoid)

# backward() needs one final value, so sum the outputs first
sigmoid_output.sum().backward()  # stores each input gradient in x_sigmoid.grad

print(" input | output | gradient")
for value, output, gradient in zip(
    x_sigmoid.detach(),
    sigmoid_output.detach(),
    x_sigmoid.grad,
):
    print(f"{value.item():>6.1f} | {output.item():.4f} | {gradient.item():.6f}")

At `-10` and `10`, the sigmoid gradients are close to zero. Those gradients would create very small weight updates during backpropagation.

## ReLU

```text
ReLU(x) = max(0, x)
```

- Negative input: output `0`, gradient `0`
- Positive input: output equals input, gradient `1`
- No upper bound

The gradient stays `1` for positive values, which helps gradients travel through deep networks. ReLU is a reliable default for hidden layers.

In [ ]:
relu = nn.ReLU()
x_relu = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0], requires_grad=True)
relu_output = relu(x_relu)

relu_output.sum().backward()  # calculate and store ReLU's gradients

print(" input | output | gradient")
for value, output, gradient in zip(
    x_relu.detach(),
    relu_output.detach(),
    x_relu.grad,
):
    print(f"{value.item():>6.1f} | {output.item():>6.2f} | {gradient.item():.2f}")

### Dying ReLU

If a neuron always receives negative inputs:

```text
negative input -> output 0 -> gradient 0 -> no update
```

The neuron can stop learning. This is called a **dying ReLU**.

## Leaky ReLU

Leaky ReLU keeps a small slope for negative inputs.

```text
LeakyReLU(x) = x       when x > 0
               0.01x   when x <= 0
```

`negative_slope` controls the negative side. PyTorch's default is `0.01`.

In [ ]:
leaky_relu = nn.LeakyReLU(negative_slope=0.01)
x_leaky = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0], requires_grad=True)
leaky_output = leaky_relu(x_leaky)

leaky_output.sum().backward()  # negative inputs keep a small gradient

print(" input | output | gradient")
for value, output, gradient in zip(
    x_leaky.detach(),
    leaky_output.detach(),
    x_leaky.grad,
):
    print(f"{value.item():>6.1f} | {output.item():>6.2f} | {gradient.item():.2f}")

## Using Them in a Model

Place the activation **between** linear layers.

In [ ]:
relu_model = nn.Sequential(
    nn.Linear(7, 8),
    nn.ReLU(),             # hidden-layer activation
    nn.Linear(8, 3),       # raw scores for 3 classes
)

leaky_relu_model = nn.Sequential(
    nn.Linear(7, 8),
    nn.LeakyReLU(negative_slope=0.01),  # lets negative gradients pass
    nn.Linear(8, 3),
)

## Quick Look

| Activation | Negative input | Positive input | Common use |
|---|---:|---:|---|
| Sigmoid | approaches `0` | approaches `1` | Binary output |
| ReLU | `0` | unchanged | Default hidden activation |
| Leaky ReLU | `0.01 * x` | unchanged | Hidden activation when dying ReLU is a concern |

```python
nn.ReLU()
nn.LeakyReLU(negative_slope=0.01)
```